# 03 · 优化与回测框架

**目标**：跑 Walk-Forward 滚动样本外（WFA）主循环，对每个调仓日用过去 60 个月数据求最优权重，下一调仓日前持仓记录 OOS 收益。

**输入**：`prices_daily.parquet`、`returns_daily.parquet`、`universe.csv`

**输出**：
- `wfa_results.csv`（每个调仓日的 IS/OOS 指标 + 衰减率）
- `weight_history.csv`（每个调仓日每只 ETF 的权重）
- `oos_returns.csv`（OOS 期间组合日收益）

In [ ]:
# ============================================================
# cell 0: imports + 全局参数
# ============================================================
from jqdata import *            # 聚宽 magic
import sys, os, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from datetime import date

import pandas as pd
import numpy as np

PROJ = Path('/Users/huhao/src/codesnip/python/ai/028-jukuan').resolve()
sys.path.insert(0, str(PROJ))

from etf_portfolio.data_loader import load_parquet
from etf_portfolio.wfa import walk_forward, score_robustness
from etf_portfolio.risk_parity import risk_parity
from etf_portfolio.metrics import full_metrics, decay_rate

# 全局参数
TRAIN_MONTHS   = 60         # 训练窗口
TEST_MONTHS    = 1          # OOS 推进步长
W_MAX          = 0.30       # 单只权重上限
HALFLIFE       = 120        # EWM 半衰期
RF             = 0.025      # 无风险利率（年化）
OBJECTIVES     = ('sharpe', 'calmar', 'minvar')

OUTPUT_DIR = PROJ / 'etf_portfolio' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
# ============================================================
# cell 1: 加载候选池价格
# ============================================================
prices = load_parquet(OUTPUT_DIR / 'prices_daily.parquet')
uni = pd.read_csv(OUTPUT_DIR / 'universe.csv', dtype={'code': str})
print(f'价格矩阵: {prices.shape}, 候选池: {len(uni)} 只')
print(f'日期范围: {prices.index.min().date()} ~ {prices.index.max().date()}')
print(f'有效 ETF 比例: {prices.notna().mean().mean():.2%}')

In [ ]:
# ============================================================
# cell 2: 单窗口优化演示（验证优化器正常工作）
# ============================================================
from etf_portfolio.covariance import ledoit_wolf_cov, estimate_mu
from etf_portfolio.optimizers import optimize_portfolio

# 用最近 60 个月的数据做演示
end = prices.index.max()
start = end - pd.DateOffset(months=TRAIN_MONTHS)
demo_prices = prices.loc[start:end].iloc[:-1].dropna(how="any")
demo_ret = demo_prices.pct_change().dropna()
print(f'演示训练期: {demo_ret.index.min().date()} ~ {demo_ret.index.max().date()}, T={len(demo_ret)}')

cov = ledoit_wolf_cov(demo_ret)
mu = estimate_mu(demo_ret, halflife=HALFLIFE)
print(f'协方差矩阵 shape={cov.shape}, 均值向量 shape={mu.shape}')

for obj in OBJECTIVES:
    if obj == 'calmar':
        w = optimize_portfolio(mu, cov, returns_for_calmar=demo_ret, objective='calmar', w_max=W_MAX)
    else:
        w = optimize_portfolio(mu, cov, objective=obj, w_max=W_MAX, rf=RF)
    print(f'\n[{obj}] Top 5 权重:')
    s = pd.Series(w, index=prices.columns).sort_values(ascending=False)
    print(s.head().round(4))

In [ ]:
# ============================================================
# cell 3: 风险平价对照组（一次性）
# ============================================================
w_rp = risk_parity(cov.values)
rp_weights = pd.Series(w_rp, index=prices.columns).sort_values(ascending=False)
print('[risk_parity] Top 5 权重:')
print(rp_weights.head().round(4))

In [ ]:
# ============================================================
# cell 4: WFA 主循环（滚动样本外）
# ============================================================
wfa = walk_forward(
    prices,
    objectives=OBJECTIVES,
    train_months=TRAIN_MONTHS,
    test_months=TEST_MONTHS,
    w_max=W_MAX,
    halflife=HALFLIFE,
    rf=RF,
    show_progress=True,
)

In [ ]:
# ============================================================
# cell 5: 查看 WFA 结果概览
# ============================================================
print(f'WFA 共 {len(wfa.metrics)} 条记录（{len(OBJECTIVES)} 目标 × {len(wfa.metrics)//len(OBJECTIVES)} 次滚动）')
wfa.metrics.head(10).round(3)

In [ ]:
# ============================================================
# cell 6: 衰减率分布（按目标分组）
# ============================================================
summary = wfa.metrics.groupby('objective').agg(
    n=('decay', 'size'),
    is_sharpe_mean=('is_sharpe', 'mean'),
    oos_sharpe_mean=('oos_sharpe', 'mean'),
    decay_median=('decay', 'median'),
    decay_p75=('decay', lambda x: x.quantile(0.75)),
    oos_mdd_mean=('oos_mdd', 'mean'),
    oos_annret_mean=('oos_annret', 'mean'),
).round(4)
summary

In [ ]:
# ============================================================
# cell 7: 持久化
# ============================================================
wfa.metrics.to_csv(OUTPUT_DIR / 'wfa_results.csv', index=False, encoding='utf-8-sig')
wfa.weights.to_csv(OUTPUT_DIR / 'weight_history.csv', index=False, encoding='utf-8-sig')
if not wfa.oos_returns.empty:
    wfa.oos_returns.to_csv(OUTPUT_DIR / 'oos_returns.csv', encoding='utf-8-sig')
print('已写入：')
print('  wfa_results.csv   (IS/OOS 指标 + 衰减率)')
print('  weight_history.csv (每调仓日每 ETF 权重)')
print('  oos_returns.csv    (OOS 期间组合日收益)')

In [ ]:
# ============================================================
# cell 8: 衰减率分布直方图
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, obj in zip(axes, OBJECTIVES):
    sub = wfa.metrics[wfa.metrics['objective'] == obj]['decay'].dropna()
    ax.hist(sub, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
    ax.axvline(0.30, color='green', linestyle='--', linewidth=1, label='优秀线 0.30')
    ax.axvline(0.60, color='orange', linestyle='--', linewidth=1, label='尚可线 0.60')
    ax.axvline(0.70, color='red', linestyle='--', linewidth=1, label='过拟合线 0.70')
    ax.set_title(f'{obj}')
    ax.set_xlabel('衰减率')
    ax.set_ylabel('频次')
    ax.legend(fontsize=8)
fig.suptitle('WFA 衰减率分布（按目标）', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'decay_distribution.png', dpi=120)
plt.show()

In [ ]:
# ============================================================
# cell 9: 稳健度评分（路线 B 核心）
# ============================================================
score = score_robustness(wfa.weights)
uni_lookup = dict(zip(uni['code'], uni['name']))
score_named = score.copy()
score_named.index = [f'{c} ({uni_lookup.get(c, c)})' for c in score.index]
print('稳健度评分 Top 15：')
score_named.head(15).round(4)

In [ ]:
# 持久化稳健度评分
score_named.to_csv(OUTPUT_DIR / 'robustness_score.csv', encoding='utf-8-sig')
print(f'已写入 robustness_score.csv')

## 中间结论

- WFA 主循环已跑完，输出每个调仓日的 IS/OOS 指标 + 衰减率
- 稳健度评分表显示哪些 ETF 反复被赋予可观权重（路线 B 候选）
- 衰减率分布应集中在 < 60%；若严重偏离说明目标函数或窗口选择有问题
- 进入 `04_稳健组合推荐.ipynb` 组合最终基线